In [2]:
import pandas as pd
import numpy as np

# Load data

In [6]:
df_eq_all_infos = pd.read_parquet("data/transformed_data/df_eq_all_infos.parquet")
df_exp_abs_emissions_growth_rate_stats = pd.read_parquet("data/transformed_data/exp_abs_emissions_growth_rate_stats.parquet")
df_hist_abs_emissions_growth_rate_stats = pd.read_parquet("data/transformed_data/hist_abs_emissions_growth_rate_stats.parquet")
df_hist_intensities_growth_rate_stats = pd.read_parquet("data/transformed_data/hist_intensities_growth_rate_stats.parquet")
df_hist_emissions = pd.read_parquet('data/transformed_data/hist_abs_emissions.parquet')
df_hist_intensities = pd.read_parquet('data/transformed_data/hist_intensities.parquet')
df_abs_em_rate = pd.read_parquet('data/transformed_data/hist_abs_emissions_growth_rate.parquet')



In [15]:
df = pd.read_parquet('output/df_merged_all_infos.parquet')
scopes = ["s1", "s2", "s3_d", "s3_u"]
df = df[df['scope'].isin(scopes)]
df["scope"] = pd.Categorical(df["scope"], categories=scopes, ordered=True)

In [16]:
df_abs_em_rate = pd.read_parquet('data/transformed_data/hist_abs_emissions_growth_rate.parquet')
scopes = ["s1", "s2", "s3_d", "s3_u"]
df_abs_em_rate = df_abs_em_rate[df_abs_em_rate['scope'].isin(scopes)]
df_abs_em_rate["scope"] = df_abs_em_rate["scope"].cat.set_categories(scopes)

years_str = sorted(
    [c for c in df_abs_em_rate.columns if c.isdigit()],
    key=int, reverse=False)

df_eq_abs_em_rate = pd.merge(df[['isin', 'scope','high_impact_sector', 'region_0']], df_abs_em_rate, how='left', on=['isin', 'scope'])

delta_years_str = [f"{int(y)-1}-{y}" for y in years_str]

df_abs_em_rate = df_abs_em_rate.rename(columns={y: f"{int(y)-1}-{y}" for y in years_str})

In [17]:
delta_years_str

['2019-2020', '2020-2021', '2021-2022', '2022-2023']

In [13]:
years_str

[]

# Parameters

In [13]:
years_str = [str(year) for year in range(2020,2024)]

In [14]:
scopes = ['s1', 's2', 's3']
hist_abs_emissions_growth_rate = hist_abs_emissions_growth_rate[hist_abs_emissions_growth_rate['scope'].isin(scopes)]
hist_intensities_growth_rate = hist_intensities_growth_rate[hist_intensities_growth_rate['scope'].isin(scopes)]
hist_abs_emissions_growth_rate["scope"] = hist_abs_emissions_growth_rate["scope"].cat.set_categories(scopes)
hist_intensities_growth_rate["scope"] = hist_intensities_growth_rate["scope"].cat.set_categories(scopes)

In [15]:
hist_abs_emissions_growth_rate['scope'].unique()

['s1', 's2', 's3']
Categories (3, object): ['s1' < 's2' < 's3']

# Coverage

In [16]:
print('nbr eq from iss emissions:', hist_abs_emissions_growth_rate['isin'].nunique())
print('nbr eq with intensities:', hist_intensities_growth_rate['isin'].nunique())
print('nbr eq with emissions and intensities:', pd.merge(
    hist_abs_emissions_growth_rate, hist_intensities_growth_rate, 
    on=['isin', 'scope'], how='inner')['isin'].nunique())
print('nbr eq in our universe:', df_eq_all_infos['isin'].nunique())

nbr eq from iss emissions: 50321
nbr eq with intensities: 50321
nbr eq with emissions and intensities: 50321
nbr eq in our universe: 38125


In [27]:
merged_infos_eq_emissions = pd.merge(df_eq_all_infos, hist_abs_emissions_growth_rate, on='isin', how='left')
merged_infos_eq_emissions.groupby('scope', observed=False)[years_str].count()


,2020,2021,2022,2023
scope,,,,
s1,6053,6342,6611,6646
s2,6054,6328,6576,6608
s3,6082,6380,6660,6730


In [28]:
merged_infos_eq_intensities = pd.merge(df_eq_all_infos, hist_intensities_growth_rate, on='isin', how='left')
merged_infos_eq_intensities.groupby('scope', observed=False)[years_str].count()

,2020,2021,2022,2023
scope,,,,
s1,5553,6016,6213,6248
s2,5556,6001,6178,6206
s3,5580,6050,6259,6321


# Outsiders

In [22]:
def compute_prop_outsider_by_scope(df, threshold):
    group_scope = df.groupby("scope", observed=True)[years_str]
    n_sup_threshold = group_scope.apply(lambda x: (x>threshold).sum())
    n_total = group_scope.count()
    prop = n_sup_threshold / n_total
    df_prop = n_sup_threshold.astype(str) + "/" + n_total.astype(str) + " (" + prop.map(lambda v: f"{v:.1%}") + ")"   
    return df_prop 

### On ISS universe

In [23]:
threshold = 0.1
compute_prop_outsider_by_scope(hist_abs_emissions_growth_rate, threshold)

,2020,2021,2022,2023
scope,,,,
s1,5483/27729 (19.8%),9966/27190 (36.7%),9353/26856 (34.8%),7643/32159 (23.8%)
s2,5502/27720 (19.8%),9632/27183 (35.4%),9197/26838 (34.3%),7308/32115 (22.8%)
s3,20289/27776 (73.0%),17131/27271 (62.8%),12127/26991 (44.9%),8860/32482 (27.3%)


In [24]:
threshold = 0.2
compute_prop_outsider_by_scope(hist_abs_emissions_growth_rate, threshold)

,2020,2021,2022,2023
scope,,,,
s1,3697/27729 (13.3%),6822/27190 (25.1%),7889/26856 (29.4%),5187/32159 (16.1%)
s2,3711/27720 (13.4%),6654/27183 (24.5%),7784/26838 (29.0%),4961/32115 (15.4%)
s3,19933/27776 (71.8%),13938/27271 (51.1%),9291/26991 (34.4%),6275/32482 (19.3%)


### On our universe

In [25]:
threshold = 0.1
compute_prop_outsider_by_scope(merged_infos_eq_emissions, threshold)

,2020,2021,2022,2023
scope,,,,
s1,1443/6053 (23.8%),2648/6342 (41.8%),2192/6611 (33.2%),1837/6646 (27.6%)
s2,1475/6054 (24.4%),2308/6328 (36.5%),1902/6576 (28.9%),1759/6608 (26.6%)
s3,4577/6082 (75.3%),4062/6380 (63.7%),3592/6660 (53.9%),2190/6730 (32.5%)


In [26]:
threshold = 0.2
compute_prop_outsider_by_scope(merged_infos_eq_emissions, threshold)

,2020,2021,2022,2023
scope,,,,
s1,1006/6053 (16.6%),1860/6342 (29.3%),1706/6611 (25.8%),1266/6646 (19.0%)
s2,1018/6054 (16.8%),1679/6328 (26.5%),1468/6576 (22.3%),1204/6608 (18.2%)
s3,4513/6082 (74.2%),3253/6380 (51.0%),2740/6660 (41.1%),1519/6730 (22.6%)


# Replace ousiders with intensities

In [ ]:
df_merge_abs_int = pd.merge(hist_abs_emissions_growth_rate, hist_intensities_growth_rate, on=['isin', 'scope'], how='left', suffixes=['','_i'])
merged_infos_eq_abs_int = pd.merge(df_eq_all_infos, df_merge_abs_int, on=['isin'], how='left') 

In [39]:
def replace_abs_em_with_intensities(df_merge_abs_int, threshold):
    # df_merge_abs_int = pd.merge(hist_abs_emissions_growth_rate, hist_intensities_growth_rate, on=['isin', 'scope'], how='left', suffixes=['','_i'])
    df_int_repl_abs = df_merge_abs_int.copy()
    for year in years_str:
        df_int_repl_abs[year] = df_int_repl_abs.apply(lambda x: np.nan if pd.isnull(x[year]) else x[f'{year}_i'] if (
            x[year]>threshold and (pd.notnull(x[f'{year}_i']))) else x[year], axis=1)
    df_int_repl_abs = df_int_repl_abs[['isin', 'scope', 'name'] + [y for y in years_str]]    
    return df_int_repl_abs

### On ISS universe

In [40]:
threshold = 0.1
df_int_repl_abs = replace_abs_em_with_intensities(df_merge_abs_int, threshold)
compute_prop_outsider_by_scope(df_int_repl_abs, threshold)

,2020,2021,2022,2023
scope,,,,
s1,4989/27729 (18.0%),8699/27190 (32.0%),8589/26856 (32.0%),7045/32159 (21.9%)
s2,5007/27720 (18.1%),8544/27183 (31.4%),8528/26838 (31.8%),6696/32115 (20.9%)
s3,20208/27776 (72.8%),15591/27271 (57.2%),10484/26991 (38.8%),7926/32482 (24.4%)


In [41]:
threshold = 0.2
df_int_repl_abs = replace_abs_em_with_intensities(df_merge_abs_int, threshold)
compute_prop_outsider_by_scope(df_int_repl_abs, threshold)

,2020,2021,2022,2023
scope,,,,
s1,3388/27729 (12.2%),5972/27190 (22.0%),7325/26856 (27.3%),4814/32159 (15.0%)
s2,3410/27720 (12.3%),5869/27183 (21.6%),7286/26838 (27.1%),4602/32115 (14.3%)
s3,19855/27776 (71.5%),12589/27271 (46.2%),8102/26991 (30.0%),5661/32482 (17.4%)


### On our universe

In [ ]:
threshold = 0.1
df_int_repl_abs = replace_abs_em_with_intensities(merged_infos_eq_abs_int, threshold)
compute_prop_outsider_by_scope(df_int_repl_abs, threshold)

,2020,2021,2022,2023
scope,,,,
s1,931/6053 (15.4%),1338/6342 (21.1%),1412/6611 (21.4%),1218/6646 (18.3%)
s2,962/6054 (15.9%),1186/6328 (18.7%),1211/6576 (18.4%),1133/6608 (17.1%)
s3,4494/6082 (73.9%),2464/6380 (38.6%),1914/6660 (28.7%),1230/6730 (18.3%)


In [43]:
threshold = 0.2
df_int_repl_abs = replace_abs_em_with_intensities(merged_infos_eq_abs_int, threshold)
compute_prop_outsider_by_scope(df_int_repl_abs, threshold)

,2020,2021,2022,2023
scope,,,,
s1,687/6053 (11.3%),978/6342 (15.4%),1127/6611 (17.0%),884/6646 (13.3%)
s2,707/6054 (11.7%),862/6328 (13.6%),957/6576 (14.6%),831/6608 (12.6%)
s3,4432/6082 (72.9%),1860/6380 (29.2%),1520/6660 (22.8%),889/6730 (13.2%)
